In [ ]:
### imports and file settings
import sys, os, re
from pathlib import Path
from functools import reduce

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from upsetplot import UpSet, from_memberships

# Automatically detect and add src folder
src_path = Path.cwd().parent / "src"
if str(src_path) not in sys.path:
    sys.path.append(str(src_path))

# Now import the modules from the src directory
import percolator_file_functions as pff
import fdr_functions as fdrf


###
### parameters
###
# basepath: the path for same datasets, inside could be several databases (subfolders)
basepath = "/mnt/data/pipeline-of-identifications/publication/MSV000095036-cancer-array/"

# output files and paths
results_path = "./results-MSV000095036-cancer-array/"


# databases: list of databases, basically subpaths
databases = ["sprot", "proteome"]#, "prohap"]
#databases = ["sprot", "proteome"]
#databases = ["DB1", "DB2"]

entrapment_path = "proteome-entrap"
#entrapment_path = "DB1-entrap"

files_pattern = re.compile(r".*")
#files_pattern = re.compile(r"S9.*")

entr_fold = 3

filter_cutoff = 0.01

idlevels = ["psms", "peptidoform"]

# searchengines can be defined (subfolders under DBs)
#searchengines = ["comet", "maxquant", "msamanda", "msfragger", "msgfplus", "sage", "xtandem"]
searchengines = ["comet", "maxquant", "msamanda", "msfragger", "sage", "xtandem"]

nr_histo_bins = 50

### -----------------------------------------------------
### below here, no further settings should be necessary

se_to_basescore = {
    "comet": "neg_ln_comet_expectation_value",
    "maxquant": "maxquant_Score",
    "msamanda": "amanda_score",
    "msfragger": "neg_ln_msfragger_expect",
    "msgfplus": "ln_msgf_specevalue",
    "sage": "sage_discriminant_score",
    "xtandem": "neg_ln_xtandem_expect"
}

postprocess_levels = ["tda", "percolator", "ms2rescore", "oktoberfest"]

# create results_path if not existing
if not os.path.exists(results_path):
    os.makedirs(results_path)


In [ ]:
# look for files in the folders
initial_search_pin_suffix = "psm_utils.pin"
percolator_out_suffix = "psm_utils.pout"
ms2rescore_percolator_pin_suffix = "ms2rescore.corrected.pin"
ms2rescore_percolator_out_suffix = "ms2rescore.corrected.pout"
oktoberfest_percolator_pin_suffix = "features.oktoberfest.pin"
oktoberfest_percolator_out_suffix = "features.oktoberfest.pout"

files = {}
for database in databases:
    files[database] = {}
    for searchengine in searchengines:
        path = f"{basepath}/{database}/{searchengine}/"

        lst = os.listdir(path)
        lst = [ f for f in lst if files_pattern.match(f) ]
        lst.sort()

        files[database][searchengine] = {}
        files[database][searchengine]["initial_search_pin"]                 = [f"{path}/{f}" for f in lst if f.endswith(initial_search_pin_suffix)]
        files[database][searchengine]["percolator_out"]                     = [f"{path}/{f}" for f in lst if f.endswith(percolator_out_suffix)]
        files[database][searchengine]["ms2rescore_percolator_pin"]          = [f"{path}/{f}" for f in lst if f.endswith(ms2rescore_percolator_pin_suffix)]
        files[database][searchengine]["ms2rescore_percolator_out"]          = [f"{path}/{f}" for f in lst if f.endswith(ms2rescore_percolator_out_suffix)]
        files[database][searchengine]["oktoberfest_percolator_pin"]          = [f"{path}/{f}" for f in lst if f.endswith(oktoberfest_percolator_pin_suffix)]
        files[database][searchengine]["oktoberfest_percolator_out"]          = [f"{path}/{f}" for f in lst if f.endswith(oktoberfest_percolator_out_suffix)]

# infer number of runs / raw files from first db / first searchengine
nr_runs = len(files[databases[0]][searchengines[0]]['initial_search_pin'])

In [ ]:
# read in the results / files

# needs to set some Oktoberfest modifications and peptides to same format as other engines
OKTOBERFEST_MODIFICATION_REPLACEMENTS = [
    ("[UNIMOD:Carbamidomethyl]", "[UNIMOD:4]"),
    ("[UNIMOD:Oxidation]", "[UNIMOD:35]"),
]

def replace_oktoberfest_modifications(sequences):
    return sequences.apply(
        lambda x: x.strip("_")
    ).apply(
        lambda seq: reduce(
            lambda seq_x, repl: seq_x.replace(repl[1], repl[0]), # lambda to replace
            OKTOBERFEST_MODIFICATION_REPLACEMENTS, # replacements
            seq # starting with the sequences as stated in the PSMs file
        )
    )

results = {}
for database in databases:
    print(f"reading in results of {database}")
    results[database] = {}
    for searchengine in searchengines:
        print(f"\treading in searchengine {searchengine}")
        results[database][searchengine] = {
            "tda": [],
            "percolator": [],
            "ms2rescore": [],
            "oktoberfest": []
        }

        for rep in range(nr_runs):
            print(f"\t\treading in run {rep}")
            # read in the initial search results (from PIN)
            se_score = se_to_basescore[searchengine]
            pin_file = files[database][searchengine]["initial_search_pin"][rep]
            pin_df = pff.parse_percolator_pin(pin_file)
            pin_df[se_score] = pd.to_numeric(pin_df[se_score])
            # calculate the FDR
            pin_df = fdrf.calculate_fdr(pin_df, se_score)
            results[database][searchengine]["tda"].append(pin_df)

            # read in the percolator results
            pout_file = files[database][searchengine]["percolator_out"][rep]
            pout_df = pff.read_enriched_pout_with_pin_data(pout_file, pin_file)
            results[database][searchengine]["percolator"].append(pout_df)

            # read in the percolator after MS2Rescore results
            ms2rescore_pin_file = files[database][searchengine]["ms2rescore_percolator_pin"][rep]
            ms2rescore_pout_file = files[database][searchengine]["ms2rescore_percolator_out"][rep]
            ms2rescore_pout_df = pff.read_enriched_pout_with_pin_data(ms2rescore_pout_file, ms2rescore_pin_file)
            results[database][searchengine]["ms2rescore"].append(ms2rescore_pout_df)

            # read in the percolator after Oktoberfest results
            oktoberfeste_pin_file = files[database][searchengine]["oktoberfest_percolator_pin"][rep]
            oktoberfeste_pout_file = files[database][searchengine]["oktoberfest_percolator_out"][rep]
            oktoberfest_pout_df = pff.read_enriched_pout_with_pin_data(oktoberfeste_pout_file, oktoberfeste_pin_file)
            oktoberfest_pout_df["peptide"] = replace_oktoberfest_modifications(oktoberfest_pout_df["peptide"])
            results[database][searchengine]["oktoberfest"].append(oktoberfest_pout_df)


In [ ]:
# Target decoy distributions like in "Quality Control for the Target Decoy Approach for Peptide Identification"
# https://pubs.acs.org/doi/10.1021/acs.jproteome.2c00423

# boxplots
for db in databases:
    # Create a matrix of PSM plots: rows=databases, cols=postprocess_levels
    fig, axes = plt.subplots(len(searchengines), nr_runs, figsize=(6*nr_runs, 4*len(searchengines)), sharey='row', sharex='row')

    for rep in range(nr_runs):
        for j_se, searchengine in enumerate(searchengines):

            if nr_runs == 1:
                ax = axes[j_se]
            else:
                ax = axes[j_se, rep]
            
            df_rep = results[db][searchengine]['tda'][rep]
            se_score = se_to_basescore[searchengine]

            bin_range = [np.percentile(df_rep[se_score], 0.5), np.percentile(df_rep[se_score], 99.5)]

            targets = df_rep[df_rep["Label"] == 1][se_score]
            decoys = df_rep[df_rep["Label"] == -1][se_score]

            ax.hist(targets, bins=nr_histo_bins, range=bin_range, histtype='bar', label="targets", color='green', edgecolor='black')
            ax.hist(decoys, bins=nr_histo_bins, range=bin_range, histtype='bar', label="decoys", color='darkorange', edgecolor='black', alpha=0.66)

            ax.set_xlabel("score")

            if j_se == 0:
                ax.set_title(f"run {rep}")
            if rep == nr_runs - 1 and j_se == len(searchengines) - 1:
                ax.legend()
            if rep == 0:
                ax.set_ylabel(searchengine)
            
    plt.tight_layout()

    # plot to svg and png file
    plt.savefig(f"{results_path}/target_decoy_distributions_{db}.svg", format='svg', bbox_inches='tight')
    plt.savefig(f"{results_path}/target_decoy_distributions_{db}.png", format='png', bbox_inches='tight')

    plt.show()

# pp-plots
for i_db, db in enumerate(databases):
    for rep in range(nr_runs):
        # pp-plots
        plt.figure(figsize=(8, 6))

        plt.plot([0, 1], [0, 0], color='black', label=None)
        plt.title(f"scaled P-P plot for {db}, run {rep}")
        plt.xlabel('Fd')
        plt.ylabel('Ft-pi0')

        for j_se, searchengine in enumerate(searchengines):

            df_rep = results[db][searchengine]['tda'][rep]

            # TODO: move this into a dict
            if searchengine == "comet":
                se_score = "neg_ln_comet_expectation_value"
            elif searchengine == "maxquant":
                se_score = "maxquant_Score"
            elif searchengine == "msamanda":
                se_score = "amanda_score"
            elif searchengine == "msfragger":
                se_score = "neg_ln_msfragger_expect"
            elif searchengine == "msgfplus":
                se_score = "ln_msgf_specevalue"
            elif searchengine == "sage":
                se_score = "sage_discriminant_score"
            elif searchengine == "xtandem":
                se_score = "neg_ln_xtandem_expect"

            pp_sorted = df_rep[['Label', se_score]]
            pp_sorted.columns = ['Label', 'score']

            pp_sorted= pp_sorted.sort_values(by='score')

            nr_targets = sum(pp_sorted['Label'] == 1)
            nr_decoys = sum(pp_sorted['Label'] == -1)

            pp_sorted['Ft'] = pp_sorted['Label'].apply(lambda x: 1 if x == 1 else 0).cumsum() / nr_targets
            pp_sorted['Fd'] = pp_sorted['Label'].apply(lambda x: 1 if x == -1 else 0).cumsum() / nr_decoys
            
            pi0_slope = nr_decoys / nr_targets
            pp_sorted['Ft-pi0'] = pp_sorted['Ft'] - pp_sorted['Fd']*pi0_slope

            plt.plot(pp_sorted['Fd'], pp_sorted['Ft-pi0'], label=searchengine)
        
        plt.legend()

        # plot to svg and png file
        plt.savefig(f"{results_path}/scaled_pp_{db}_{rep}.svg", format='svg', bbox_inches='tight')
        plt.savefig(f"{results_path}/scaled_pp_{db}_{rep}.png", format='png', bbox_inches='tight')

        plt.show()

In [ ]:
#### calculate counting results, give text output and plots

filenames = [os.path.basename(f).split('.', 1)[0] for f in files[databases[0]][searchengines[0]]['initial_search_pin']]

for idlevel in idlevels:
    # Create a MultiIndex DataFrame to store the results
    index = pd.MultiIndex.from_product([postprocess_levels, searchengines], names=["postprocess_level", "searchengine"])

    df_columns = []
    for db in databases:
        df_columns.append(db)
        df_columns.append(f"{db} stdev")
        for name in filenames:
            df_columns.append(name)
        
    results_df = pd.DataFrame(index=index, columns=df_columns)

    for postprocess_level in postprocess_levels:
        for se in searchengines:
            for db in databases:
                rep_results = []
                for rep in range(nr_runs):
                    df_rep = results[db][se][postprocess_level][rep]
                    filtered = df_rep[df_rep['q-value'] <= filter_cutoff]

                    peprowname = "peptide"
                    if (postprocess_level == "tda"):
                        peprowname = "Peptide"
                        filtered = filtered[filtered["is_decoy"] == False]

                    if (idlevel == "psms"):
                        rep_results.append(filtered.shape[0])
                    elif (idlevel == "peptidoform"):
                        nr_peptidoforms = filtered[peprowname].apply(lambda x: x.replace("I", "J").replace("L", "J").replace("UNJMOD", "UNIMOD")).unique().shape[0]
                        rep_results.append(nr_peptidoforms)
                    
                    results_df.loc[(postprocess_level, se), filenames[rep]] = rep_results[rep]

                results_df.loc[(postprocess_level, se), db] = np.mean(rep_results)
                results_df.loc[(postprocess_level, se), f"{db} stdev"] = np.std(rep_results)

    # write the table to file
    results_df.to_csv(f"{results_path}/{idlevel}_overview.tsv", sep="\t")

    # Plot bar charts for each database from results_df
    for db in databases:
        means = results_df[db].astype(float)
        stds = results_df[f"{db} stdev"].astype(float)

        # Unstack to get postprocess_level as columns, searchengine as index
        means_unstacked = means.unstack(level=-1, sort=False)
        stds_unstacked = stds.unstack(level=-1, sort=False)

        ax = means_unstacked.plot.bar(yerr=stds_unstacked, figsize=(12, 6), capsize=4)
        ax.set_title(f"{idlevel} identifications for {db}")
        ax.set_ylabel("Number of Identifications")
        ax.set_xlabel("postprocess level")
        ax.set_xticklabels(means_unstacked.index, rotation=45, ha='right')
        plt.tight_layout()

        # plot to svg and png file
        plt.savefig(f"{results_path}/{idlevel}_{db}_identifications.svg", format='svg', bbox_inches='tight')
        plt.savefig(f"{results_path}/{idlevel}_{db}_identifications.png", format='png', bbox_inches='tight')

        plt.show()
        plt.close()


In [ ]:
# Plot pseudo ROC curves

# Create a matrix of PSM plots: rows=databases, cols=postprocess_levels
fig, axes = plt.subplots(len(databases), len(postprocess_levels), figsize=(6*len(postprocess_levels), 4*len(databases)), sharey=True)

for i_db, db in enumerate(databases):
    for j_pp, postprocess_level in enumerate(postprocess_levels):
        if len(databases) == 1:
            ax = axes[j_pp]
        else:
            ax = axes[i_db, j_pp]
        for se in searchengines:
            # Collect cumulative curves for all replicates
            cumulative_curves = []
            qvalue_grids = []

            for rep in range(nr_runs):
                df_rep = results[db][se][postprocess_level][rep]
                filtered_df_rep = df_rep[df_rep["q-value"] <= filter_cutoff]
                if filtered_df_rep.empty:
                    continue
                df_counted_rep = filtered_df_rep.sort_values("q-value").groupby(by=["q-value"]).count()
                df_counted_rep['cumulative'] = df_counted_rep.iloc[:, 0].cumsum()
                df_counted_rep['cumulative_norm'] = df_counted_rep['cumulative'] / df_counted_rep['cumulative'].max()
                qvalue_grids.append(df_counted_rep.index.values)
                cumulative_curves.append(df_counted_rep['cumulative'].values)

            # Interpolate all curves to a common q-value grid for averaging
            if cumulative_curves:
                # Define a common grid (union of all q-values)
                q_grid = np.unique(np.concatenate(qvalue_grids))
                interpolated_curves = []
                for qv, cum in zip(qvalue_grids, cumulative_curves):
                    interp = np.interp(q_grid, qv, cum)
                    interpolated_curves.append(interp)
                interpolated_curves = np.array(interpolated_curves)
                mean_curve = np.mean(interpolated_curves, axis=0)
                std_curve = np.std(interpolated_curves, axis=0)
                ax.plot(q_grid, mean_curve, label=f"{se}")
                ax.fill_between(q_grid, mean_curve - std_curve, mean_curve + std_curve, alpha=0.2)
        
        if i_db == 0:
            ax.set_title(postprocess_level)
        if j_pp == 0:
            ax.set_ylabel(db)
        if i_db == len(databases) - 1:
            ax.set_xlabel("q-value")
        if i_db == len(databases) - 1 and j_pp == len(postprocess_levels) - 1:
            ax.legend()
plt.tight_layout()

# plot to svg and png file
plt.savefig(f"{results_path}/psms_pseudoroc.svg", format='svg', bbox_inches='tight')
plt.savefig(f"{results_path}/psms_pseudoroc.png", format='png', bbox_inches='tight')


In [ ]:
### some functions for later

def upset_all_counts_contents(element_sets, nr_runs, id_column="id"):
    df = None
    for rep in range(nr_runs):
        cat_series = [
            pd.Series(True, index=list(elements[rep]), name=name)
            for name, elements in element_sets.items()
        ]
        if not all(s.index.is_unique for s in cat_series):
            raise ValueError("Got duplicate ids in a category")

        df_run = pd.concat(cat_series, axis=1, sort=False)
        if id_column in df_run.columns:
            raise ValueError("A category cannot be named %r" % id_column)
        df_run["rep"] = rep
        
        if df is None:
            df = df_run
        else:
            df = pd.concat([df, df_run])
        
    df.fillna(False, inplace=True)
    cat_names = [name for name in list(df.columns) if name != "rep"]
    df.index.name = id_column

    # #return this
    return df.reset_index().set_index(cat_names)


In [ ]:
#############
# same engine, differing DBs

min_subset_size = 100

for searchengine in searchengines:
    # Collect all PSM sets across replicates
    categories = []
    psm_sets = {}
    for db in databases:
        for postprocessing in postprocess_levels:
            if (postprocessing == "tda"):
                peprowname = "Peptide"
            else:
                peprowname = "peptide"

            # Collect PSM sets for each replicate
            replicate_sets = []
            for rep in range(nr_runs):
                df_rep = results[db][searchengine][postprocessing][rep]
                df_rep = df_rep[df_rep["q-value"] <= filter_cutoff]
                psm_set = df_rep[peprowname].apply(lambda x: x.replace("I", "J").replace("L", "J").replace("UNJMOD", "UNIMOD")).unique()
                replicate_sets.append(set(psm_set))

            # add the replicate sets
            psm_sets[f"{db}-{postprocessing}"] = replicate_sets

            categories.append(f"{db}-{postprocessing}")

    # Build a DataFrame for UpSet plot
    upset_data_intermediate = upset_all_counts_contents(psm_sets, nr_runs=nr_runs)

    idx_names = upset_data_intermediate.index.names
    counted_data = upset_data_intermediate.groupby(level=idx_names).size().to_frame(name='count') / nr_runs

    # get the memberships
    memberships = counted_data.apply(lambda x: [idx_names[i] for i in np.where(x.name)[0]], axis=1).to_list()

    # create object for UpSetPlot
    upset_data = from_memberships(memberships, data=counted_data)

    # filter later by 1% of highest intersection
    min_subset_size = int(upset_data.sort_values(by="count", ascending=False).iloc[0] / 100)

    # plot it
    plt.figure(figsize=(8, 4))
    up = UpSet(upset_data, subset_size='sum', sum_over="count", show_counts=True, min_subset_size=min_subset_size, sort_by="-degree", sort_categories_by="cardinality")

    # setting colors
    # by postprocessing included
    up.style_subsets(min_degree=1, facecolor="blue")
    up.style_subsets(min_degree=2, facecolor="purple")
    up.style_subsets(min_degree=(len(databases) * len(postprocess_levels)), facecolor="red")

    up.plot()
    plt.title(f"Peptidoforms per databases ({searchengine}, mean of runs)")

    # plot to svg and png file
    plt.savefig(f"{results_path}/overlap_databases_{searchengine}.svg", format='svg', bbox_inches='tight')
    plt.savefig(f"{results_path}/overlap_databases_{searchengine}.png", format='png', bbox_inches='tight')

    plt.show()
    plt.close()


In [ ]:
#############
# Overlap of all searchengines per database and post-processing

for upset_db in databases:
    for upset_postprocess in postprocess_levels:
        for upset_idlevel in idlevels:

            elements_se_reps = {se: [] for se in searchengines}
            for searchengine, postprocessed_data in results[upset_db].items():
                data = postprocessed_data[upset_postprocess]
                for i in range(nr_runs):
                    # filter by q-value and set the proteoform columns
                    filtered = data[i][data[i]["q-value"] <= filter_cutoff]
                    peprowname = "peptide"
                    if (upset_postprocess == "tda"):
                        peprowname = "Peptide"
                        filtered = filtered[filtered["is_decoy"] == False]

                    id_set = None
                    if (upset_idlevel == "psms"):
                        id_set = set(filtered[['ScanNr', peprowname]].apply(lambda x: f"{x['ScanNr']}:{x[peprowname].replace("I", "J").replace("L", "J").replace("UNJMOD", "UNIMOD")}", axis=1).unique())
                    elif (upset_idlevel == "peptidoform"):
                        id_set = set(filtered[peprowname].apply(lambda x: f"{x.replace("I", "J").replace("L", "J").replace("UNJMOD", "UNIMOD")}").unique())

                    elements_se_reps[searchengine].append(id_set)

            # Build a DataFrame for UpSet plot
            upset_data_intermediate = upset_all_counts_contents(elements_se_reps, nr_runs=nr_runs)
            idx_names = upset_data_intermediate.index.names
            upset_data_intermediate.reset_index(inplace=True)

            counted_data = upset_data_intermediate.groupby(by=idx_names).size().to_frame(name='count') / nr_runs

            # get the memberships
            memberships = counted_data.apply(lambda x: [idx_names[i] for i in np.where(x.name)[0]], axis=1).to_list()

            # create object for UpSetPlot
            upset_data = from_memberships(memberships, data=counted_data)

            # prepare plot
            up = UpSet(upset_data,
                    subset_size='sum',
                    sum_over="count",
                    show_counts=True,
                    sort_by="-degree",           # disables auto-sorting by cardinality
                    sort_categories_by="-input", # disables sorting categories alphabetically)
            )

            # Filter *after* initialization — affects which intersections are shown, but no longer the total numbers
            degrees = upset_data.index.to_frame().sum(axis=1)

            # align/reindex degrees to the UpSet intersections index
            aligned_degrees = degrees.reindex(up.intersections.index)    # NaN where no match
            mask = aligned_degrees.isin([len(searchengines), len(searchengines)-1, 1]).fillna(False)         # NaN -> False

            # apply the mask
            up.intersections = up.intersections[mask]

            # setting colors
            up.style_subsets(min_degree=1, facecolor="blue")
            up.style_subsets(min_degree=len(searchengines)-1, facecolor="purple")
            up.style_subsets(min_degree=len(searchengines), facecolor="red")

            # plot and titles
            up = up.plot()
            plt.suptitle(f"{upset_idlevel} in {upset_db} ({upset_postprocess})")
            up["intersections"].set_ylabel("Size of intersection (mean)")
            up["totals"].set_xlabel(f"#{upset_idlevel} (mean)")

            # plot to svg and png file
            plt.savefig(f"{results_path}/overlap_searchengines_{upset_db}_{upset_postprocess}_{upset_idlevel}.svg", format='svg', bbox_inches='tight')
            plt.savefig(f"{results_path}/overlap_searchengines_{upset_db}_{upset_postprocess}_{upset_idlevel}.png", format='png', bbox_inches='tight')

            plt.show()
            plt.close()

In [ ]:
##############################
# entrapments

entrapment_base_path = basepath + "/" + entrapment_path

def plot_fdp_intervals(postprocess_datas, label, ub_color, lb_color, max_qvalue: float=0.02):
    """
    Plot false discovery proportion (FDP) intervals for multiple runs.

    Parameters:
    postprocess_datas: list of dictionaries containing FDP data for each run
    label: string label for the plot
    ub_color: color for upper bound
    lb_color: color for lower bound
    max_qvalue: maximum q-value threshold for plotting

    Returns:
    None - displays the plot
    """
    all_q_values = []
    filtered_dfs = []

    # get all q-values
    for df_rep in postprocess_datas:
        # filter by q-value and aggregate
        filtered_df_rep = df_rep[df_rep["q-value"] <= max_qvalue].groupby('q-value').agg({
            'fdp_lb': 'min',
            'fdp_ub': 'max',
        })
        if filtered_df_rep.empty:
            continue

        all_q_values.append(filtered_df_rep.index.values)
        filtered_dfs.append(filtered_df_rep)

    # array of all q-values
    q_grid = np.unique(np.concatenate(all_q_values))

    # the list of all FDP UB and LB values
    interp_dfs = []

    for idx, rep_df in enumerate(filtered_dfs):
        # Find q-values in q_grid that don't exist in rep_df
        missing_q_values = set(q_grid) - set(rep_df.index.values)

        # Create a DataFrame with missing q-values
        missing_df = pd.DataFrame({'q-value': list(missing_q_values)})

        # Interpolate fdp_ub and fdp_lb for missing q-values
        missing_df['fdp_ub'] = np.interp(missing_df['q-value'], rep_df.index.values, rep_df['fdp_ub'])
        missing_df['fdp_lb'] = np.interp(missing_df['q-value'], rep_df.index.values, rep_df['fdp_lb'])

        # Concatenate the missing rows with rep_df
        concat_df = pd.concat([rep_df.reset_index(), missing_df], ignore_index=True)

        # Sort by q-value and add to list
        interp_dfs.append(concat_df.sort_values('q-value').reset_index(drop=True))

    fdp_aggr = pd.concat(interp_dfs).groupby('q-value').agg({
            'fdp_lb': ['min', 'max'],
            'fdp_ub': ['min', 'max'],
        })

    plt.plot(fdp_aggr['fdp_lb'], color=lb_color)
    plt.fill_between(fdp_aggr.index, fdp_aggr['fdp_lb']['min'], fdp_aggr['fdp_lb']['max'], alpha=0.2, color=lb_color, label=f"{label} / lower bound")

    plt.plot(fdp_aggr['fdp_ub'], color=ub_color)
    plt.fill_between(fdp_aggr.index, fdp_aggr['fdp_ub']['min'], fdp_aggr['fdp_ub']['max'], alpha=0.2, color=ub_color, label=f"{label} / upper bound")


for searchengine in searchengines:
    entrapment_se_path = entrapment_base_path + "/" + searchengine + "/"
    if searchengine == "comet":
        se_score = "neg_ln_comet_expectation_value"
    elif searchengine == "maxquant":
        se_score = "maxquant_Score"
    elif searchengine == "msamanda":
        se_score = "amanda_score"
    elif searchengine == "msfragger":
        se_score = "neg_ln_msfragger_expect"
    elif searchengine == "msgfplus":
        se_score = "ln_msgf_specevalue"
    elif searchengine == "sage":
        se_score = "sage_discriminant_score"
    elif searchengine == "xtandem":
        se_score = "neg_ln_xtandem_expect"

    lst = os.listdir(entrapment_se_path)
    lst.sort()

    sorted_files = {
        "percolator_pin": [],
        "percolator_pout": [],
        "ms2rescore_pin": [],
        "ms2rescore_pout": [],
        "oktoberfest_pin": [],
        "oktoberfest_pout": []
    }

    for f in lst:
        filepath = f"{entrapment_se_path}/{f}"
        if f.endswith(".psm_utils.pin"):
            sorted_files["percolator_pin"].append(filepath)
        if f.endswith(".psm_utils.pout"):
            sorted_files["percolator_pout"].append(filepath)
        if f.endswith(".ms2rescore.corrected.pin"):
            sorted_files["ms2rescore_pin"].append(filepath)
        if f.endswith(".ms2rescore.corrected.pout"):
            sorted_files["ms2rescore_pout"].append(filepath)
        if f.endswith(".features.oktoberfest.pin"):
            sorted_files["oktoberfest_pin"].append(filepath)
        if f.endswith(".features.oktoberfest.pout"):
            sorted_files["oktoberfest_pout"].append(filepath)
    
    fdp_by = "q-value"

    # Data from original search only
    tda_datas = []
    for pin_file in sorted_files["percolator_pin"]:
        pin_df = pff.parse_percolator_pin(pin_file)
        tda_df = pin_df[["Label", se_score, "Peptide"]].copy()
        tda_df["proteinIds"] = pin_df["Proteins"].copy()
        tda_df = fdrf.calculate_fdr(tda_df, se_score)
        # filter out the decoys
        tda_df = tda_df[tda_df["Label"] == 1]
        tda_df["proteinIds"] = tda_df["proteinIds"].apply(
            lambda ids: ids.split("\t") if isinstance(ids, str) else ids
        )
        tda_df = fdrf.calculate_fdp(tda_df, fdp_by=fdp_by, entr_fold=entr_fold, lowerscorebetter=True)
        tda_datas.append(tda_df)

    ## data from percolator
    percolated_datas = []
    for i in range(len(sorted_files["percolator_pout"])):
        percolated_df = pff.read_enriched_pout_with_pin_data(sorted_files["percolator_pout"][i], sorted_files["percolator_pin"][i])
        percolated_df = fdrf.calculate_fdp(percolated_df, fdp_by=fdp_by, entr_fold=entr_fold, lowerscorebetter=True)
        percolated_datas.append(percolated_df)

    ## data after rescoring
    ms2rescored_datas = []
    for i in range(len(sorted_files["ms2rescore_pout"])):
        ms2rescored_df = pff.read_enriched_pout_with_pin_data(sorted_files["ms2rescore_pout"][i], sorted_files["ms2rescore_pin"][i])
        ms2rescored_df = fdrf.calculate_fdp(ms2rescored_df, fdp_by=fdp_by, entr_fold=entr_fold, lowerscorebetter=True)
        ms2rescored_datas.append(ms2rescored_df)
    
    oktoberfested_datas = []
    for i in range(len(sorted_files["oktoberfest_pout"])):
        oktoberfested_df = pff.read_enriched_pout_with_pin_data(sorted_files["oktoberfest_pout"][i], sorted_files["oktoberfest_pin"][i])
        oktoberfested_df = fdrf.calculate_fdp(oktoberfested_df, fdp_by=fdp_by, entr_fold=entr_fold, lowerscorebetter=True)
        oktoberfested_datas.append(oktoberfested_df)

    # plotting
    plt.figure(figsize=(8, 6))

    plt.xlabel("FDR q-value")
    plt.ylabel("Estimated FDP")
    plt.title(f"FDR vs FDP for {searchengine}")
    plt.grid(True)

    plt.xlim(0, 0.02)
    plt.ylim(0, 0.02)

    # Add a diagonal line
    max_diagonal = max(max(tda_df["q-value"]), max(tda_df["fdp_ub"]), max(tda_df["fdp_lb"]))
    plt.plot([0, max_diagonal], [0, max_diagonal], color='red', linestyle='--', label='Diagonal')

    # plot the fdp intervals
    plot_fdp_intervals(tda_datas, label="TDA", ub_color='#003399', lb_color='#3399FF')
    plot_fdp_intervals(percolated_datas, label="Percolator", ub_color='#006633', lb_color='#66FF99')
    plot_fdp_intervals(ms2rescored_datas, label="MS2Rescore", ub_color='#CC6600', lb_color='#FFCC99')
    plot_fdp_intervals(oktoberfested_datas, label="Oktoberfest", ub_color='#777777', lb_color='#AAAAAA')

    # Add legend
    plt.legend()

    # plot to svg and png file
    plt.savefig(f"{results_path}/entrapment_{searchengine}.svg", format='svg', bbox_inches='tight')
    plt.savefig(f"{results_path}/entrapment_{searchengine}.png", format='png', bbox_inches='tight')

    plt.show()
